# Step 3: Feature Engineering

Feature engineering transforms raw time-series data into meaningful inputs
that help forecasting models learn trends, seasonality, and temporal patterns.


## 3.1 Base Dataset Preparation

Ensure the dataset is sorted by date and contains only relevant numeric features.


In [3]:
import pandas as pd

# Load dataset
df = pd.read_csv("../DATASET/Uber-Jan-Feb-FOIL.csv")

# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'])

# Set date as index and sort
df.set_index('date', inplace=True)
df = df.sort_index()

print("Dataset loaded successfully and df is ready")


Dataset loaded successfully and df is ready


In [4]:
# Create feature engineering base dataframe again
df_fe = df[['trips', 'active_vehicles']].copy()

df_fe.head()


,trips,active_vehicles
date,,
2015-01-01,1132,190
2015-01-01,1765,225
2015-01-01,29421,3427
2015-01-01,7679,945
2015-01-01,9537,1228


## 3.2 Time-Based Features

These features help the model understand calendar patterns and seasonality.


In [5]:
# Time-based features
df_fe['day'] = df_fe.index.day
df_fe['day_of_week'] = df_fe.index.dayofweek  # 0 = Monday
df_fe['week_of_year'] = df_fe.index.isocalendar().week.astype(int)
df_fe['month'] = df_fe.index.month
df_fe['is_weekend'] = df_fe['day_of_week'].isin([5, 6]).astype(int)

df_fe.head()


,trips,active_vehicles,day,day_of_week,week_of_year,month,is_weekend
date,,,,,,,
2015-01-01,1132,190,1,3,1,1,0
2015-01-01,1765,225,1,3,1,1,0
2015-01-01,29421,3427,1,3,1,1,0
2015-01-01,7679,945,1,3,1,1,0
2015-01-01,9537,1228,1,3,1,1,0


## 3.3 Lag Features

Lag features allow the model to learn from past trip values.
They are essential for time-series forecasting because current demand
often depends on previous days’ demand.


In [6]:
# Lag features for trips
df_fe['trips_lag_1'] = df_fe['trips'].shift(1)
df_fe['trips_lag_7'] = df_fe['trips'].shift(7)
df_fe['trips_lag_14'] = df_fe['trips'].shift(14)

print("Step 3.3 executed successfully: Lag features created")


Step 3.3 executed successfully: Lag features created


### Lag Feature Preview

The table below shows how lag features reference past trip values.


In [7]:
df_fe[['trips', 'trips_lag_1', 'trips_lag_7', 'trips_lag_14']].head(20)


,trips,trips_lag_1,trips_lag_7,trips_lag_14
date,,,,
2015-01-01,1132,NaN,NaN,NaN
2015-01-01,1765,1132.0,NaN,NaN
2015-01-01,29421,1765.0,NaN,NaN
2015-01-01,7679,29421.0,NaN,NaN
2015-01-01,9537,7679.0,NaN,NaN
2015-01-01,6903,9537.0,NaN,NaN
2015-01-02,4768,6903.0,NaN,NaN
2015-01-02,7065,4768.0,1132.0,NaN
2015-01-02,875,7065.0,1765.0,NaN


### Interpretation

- `trips_lag_1` represents the number of trips from the previous day.
- `trips_lag_7` represents the number of trips from the same day in the previous week.
- `trips_lag_14` captures bi-weekly demand patterns.

These features help the forecasting model learn short-term and weekly dependencies
in Uber trip demand.


## 3.4 Rolling Features

Rolling features smooth short-term fluctuations and help the model
capture local trends and volatility in trip demand.


In [8]:
# Rolling averages
df_fe['trips_roll_7'] = df_fe['trips'].rolling(window=7).mean()
df_fe['trips_roll_14'] = df_fe['trips'].rolling(window=14).mean()

# Rolling standard deviation (volatility)
df_fe['trips_std_7'] = df_fe['trips'].rolling(window=7).std()
print("Step 3.4 executed successfully: Rolling features created")
df_fe.head(20)


Step 3.4 executed successfully: Rolling features created


,trips,active_vehicles,day,day_of_week,week_of_year,month,is_weekend,trips_lag_1,trips_lag_7,trips_lag_14,trips_roll_7,trips_roll_14,trips_std_7
date,,,,,,,,,,,,,
2015-01-01,1132,190,1,3,1,1,0,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-01,1765,225,1,3,1,1,0,1132.0,NaN,NaN,NaN,NaN,NaN
2015-01-01,29421,3427,1,3,1,1,0,1765.0,NaN,NaN,NaN,NaN,NaN
2015-01-01,7679,945,1,3,1,1,0,29421.0,NaN,NaN,NaN,NaN,NaN
2015-01-01,9537,1228,1,3,1,1,0,7679.0,NaN,NaN,NaN,NaN,NaN
2015-01-01,6903,870,1,3,1,1,0,9537.0,NaN,NaN,NaN,NaN,NaN
2015-01-02,4768,785,2,4,1,1,0,6903.0,NaN,NaN,8743.571429,NaN,9618.976416
2015-01-02,7065,1137,2,4,1,1,0,4768.0,1132.0,NaN,9591.142857,NaN,9082.961933
2015-01-02,875,175,2,4,1,1,0,7065.0,1765.0,NaN,9464.000000,NaN,9216.024649


### Interpretation

- `trips_roll_7` captures the 7-day smoothed trend, reducing daily noise.
- `trips_roll_14` captures longer-term trend patterns.
- `trips_std_7` measures short-term variability in trip demand.

Rolling features help the model distinguish between stable trends
and sudden demand fluctuations.


## 3.5 Handling Missing Values

Lag and rolling features create missing values that must be handled before modeling.


## 3.5 Handling Missing Values

Lag and rolling features introduce missing values at the beginning of the dataset.
These rows must be handled before training any forecasting model.


In [9]:
# Check missing values before cleaning
missing_before = df_fe.isna().sum()

# Drop rows with missing values created by lag and rolling features
df_fe_clean = df_fe.dropna()

# Check missing values after cleaning
missing_after = df_fe_clean.isna().sum()

print("Step 3.5 executed successfully: Missing values handled")
print("\nMissing values before cleaning:")
print(missing_before)
print("\nMissing values after cleaning:")
print(missing_after)
print("\nFinal dataset shape:", df_fe_clean.shape)


Step 3.5 executed successfully: Missing values handled

Missing values before cleaning:
trips               0
active_vehicles     0
day                 0
day_of_week         0
week_of_year        0
month               0
is_weekend          0
trips_lag_1         1
trips_lag_7         7
trips_lag_14       14
trips_roll_7        6
trips_roll_14      13
trips_std_7         6
dtype: int64

Missing values after cleaning:
trips              0
active_vehicles    0
day                0
day_of_week        0
week_of_year       0
month              0
is_weekend         0
trips_lag_1        0
trips_lag_7        0
trips_lag_14       0
trips_roll_7       0
trips_roll_14      0
trips_std_7        0
dtype: int64

Final dataset shape: (340, 13)


## 3.6 Final Feature Set Review

This step verifies the final feature-engineered dataset before model training.
It ensures that:
- All required features are present
- No missing values remain
- The dataset is ready for forecasting models


In [10]:
# Display final columns
print("Final feature columns:")
print(df_fe_clean.columns.tolist())

# Check dataset shape
print("\nFinal dataset shape (rows, columns):")
print(df_fe_clean.shape)

# Final missing value check
print("\nMissing values check:")
print(df_fe_clean.isna().sum())

print("\nStep 3.6 executed successfully: Final feature set verified")


Final feature columns:
['trips', 'active_vehicles', 'day', 'day_of_week', 'week_of_year', 'month', 'is_weekend', 'trips_lag_1', 'trips_lag_7', 'trips_lag_14', 'trips_roll_7', 'trips_roll_14', 'trips_std_7']

Final dataset shape (rows, columns):
(340, 13)

Missing values check:
trips              0
active_vehicles    0
day                0
day_of_week        0
week_of_year       0
month              0
is_weekend         0
trips_lag_1        0
trips_lag_7        0
trips_lag_14       0
trips_roll_7       0
trips_roll_14      0
trips_std_7        0
dtype: int64

Step 3.6 executed successfully: Final feature set verified


### Interpretation

- All engineered features are present and correctly generated.
- No missing values remain in the dataset.
- The feature set includes time-based, lag, and rolling features.
- The dataset is fully prepared for forecasting model training.


# Feature Engineering Summary Report

This feature engineering phase was designed to prepare the Uber trip time-series data
for forecasting models by extracting meaningful temporal patterns and dependencies.

---

## 1. Base Dataset Preparation

A clean feature engineering base dataset was created using the following variables:
- `trips`: Daily number of Uber trips (target variable)
- `active_vehicles`: Daily number of active Uber vehicles

The dataset was indexed by date and sorted chronologically to preserve time order.

---

## 2. Time-Based Features

Calendar-based features were created to capture seasonality and temporal behavior:
- `day`: Day of the month
- `day_of_week`: Day of the week (0 = Monday)
- `week_of_year`: Week number of the year
- `month`: Month number
- `is_weekend`: Binary indicator for weekends

These features help the model learn weekly and monthly demand patterns.

---

## 3. Lag Features

Lag features were generated to capture historical dependency in trip demand:
- `trips_lag_1`: Trips from the previous day
- `trips_lag_7`: Trips from the same day in the previous week
- `trips_lag_14`: Trips from two weeks earlier

Lag features allow the model to learn short-term and weekly autocorrelation in the data.

---

## 4. Rolling Features

Rolling statistics were created to smooth fluctuations and capture local trends:
- `trips_roll_7`: 7-day rolling average of trips
- `trips_roll_14`: 14-day rolling average of trips
- `trips_std_7`: 7-day rolling standard deviation (demand volatility)

These features help distinguish stable trends from short-term variability.

---

## 5. Handling Missing Values

Lag and rolling feature creation introduced missing values at the beginning of the dataset.
Rows containing missing values were removed to ensure a clean and complete feature set
for model training.

---

## 6. Final Feature Set

The final dataset includes:
- Time-based features
- Lag features
- Rolling statistical features
- Supply-related feature (`active_vehicles`)

The resulting dataset is fully prepared for time-series forecasting models.

---

## Conclusion

The feature engineering process successfully transformed raw Uber trip data into a
model-ready dataset that captures trend, seasonality, and temporal dependency.
These engineered features provide a strong foundation for accurate demand forecasting.
